# XP Exercises: Flower Classification using CNN

This is a guided notebook for the exercises on the platform. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points appear only for key concepts that unlock intuition or transfer to other ML topics.


## What you will learn
- Building a CNN for multi class image classification
- Data loading and preprocessing with `image_dataset_from_directory`
- Image visualization techniques
- Model architecture design, compilation, and training
- Evaluating model performance with accuracy and loss plots


## What you will create
A CNN model that classifies 14 flower species.
All parts form one continuous exercise. Work through them sequentially.


## Dataset
**As stated in the exercises**  
Flower classification with 14 classes. Images are organized in class folders. A training and validation split may be provided. Images are resized to 256x256 in this notebook.

**PREFILLED info**  
This notebook expects the provided zip file to be available. The code below extracts it and locates the dataset root automatically.


In [ ]:
# PREFILLED: just execute
import os, sys, zipfile, shutil, glob, math, json, random
from pathlib import Path

DATA_ZIP = Path("./Flower Classification.zip")
EXTRACT_DIR = Path("./data/flower_data")

# Clean extract dir if re-running
if EXTRACT_DIR.exists():
    pass  # avoid deleting in case you added files; delete manually if needed
else:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Extract if a zip is present and not already extracted
if DATA_ZIP.exists():
    # Heuristically decide to extract once
    marker = EXTRACT_DIR / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        marker.write_text("ok")
        print("Extracted:", DATA_ZIP.name, "->", EXTRACT_DIR)
    else:
        print("Already extracted. Skipping.")
else:
    print("Zip file not found at", DATA_ZIP)

# Find candidate dataset roots: a dir with >= 10 subdirs assumed as classes, or contains train/val
def list_dirs(p):
    return [d for d in Path(p).iterdir() if d.is_dir()]

candidates = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    if len([d for d in Path(root).iterdir() if Path(d).is_dir()]) >= 10:
        candidates.append(Path(root))
    if "train" in [d.name.lower() for d in list_dirs(root)] and "val" in [d.name.lower() for d in list_dirs(root)]:
        candidates.append(Path(root))

candidates = sorted(set(candidates))
print("Candidate dataset roots:", [str(c) for c in candidates][:5])

## Part 1. Data exploration and visualization

**As stated in the exercises**  
Load the dataset using `image_dataset_from_directory`. Print number of images per class. Modify `visualize_images` to show a 3x3 grid for each class with the class name as the grid title. Analyze challenges you anticipate when classifying the flowers such as similar colors or shapes and intra class variation.


**Guidance**  
If a `train` or `val` folder exists, use them. Otherwise create a split from a single root with `validation_split` and `subset`. Images are resized to 256x256 RGB.


> **IMPORTANT:** we fix a low resultion for images in IMG_SIZE=(32,32) for faster training, however you can change it if you want to test out other resolutions

In [ ]:
# PREFILLED: just execute
import tensorflow as tf
from tensorflow.keras import layers

IMG_SIZE = (32, 32)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

def detect_layout(root: Path):
    root = Path(root)
    sub = [d.name.lower() for d in root.iterdir() if d.is_dir()]
    if "train" in sub and "val" in sub:
        return "provided_split", root
    return "single_root", root

# Choose a root
if 'candidates' in globals() and len(candidates) > 0:
    DS_ROOT = candidates[0]
else:
    DS_ROOT = EXTRACT_DIR  # fallback

layout, base = detect_layout(DS_ROOT)
print("Layout:", layout, "Base:", base)

In [ ]:
# PREFILLED: just execute
if layout == "provided_split":
    train_dir = next((p for p in base.iterdir() if p.name.lower()=="train"))
    val_dir   = next((p for p in base.iterdir() if p.name.lower()=="val"))
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="int"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="int"
    )
else:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        base, validation_split=0.2, subset="training", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        base, validation_split=0.2, subset="validation", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int"
    )

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", num_classes, class_names)

# Cache and prefetch
def prepare(ds):
    return ds.cache().prefetch(AUTOTUNE)

train_ds = prepare(train_ds)
val_ds = prepare(val_ds)

In [ ]:
# PREFILLED: just execute — count images per class by scanning directory
from collections import Counter
import os

def count_images_per_class(root):
    counts = {}
    for cls in class_names:
        # find folder named like cls at any depth under base
        matches = list(Path(base).rglob(cls))
        if matches:
            folder = matches[0]
            img_count = sum(1 for p in folder.rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".gif"})
            counts[cls] = img_count
        else:
            counts[cls] = None
    return counts

base = "/content/data/flower_data/Data/train"
counts = count_images_per_class(base)
counts

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_images(dataset, class_names, per_class=9):
    """Display a 3x3 grid of images for each class."""
    # Collect up to per_class images per label
    collected = {i: [] for i in range(len(class_names))}

    for images, labels in dataset.unbatch():
        label = int(labels.numpy())
        if len(collected[label]) < per_class:
            collected[label].append(images.numpy().astype('uint8'))
        if all(len(v) >= per_class for v in collected.values()):
            break

    for cls_idx, cls_name in enumerate(class_names):
        imgs = collected[cls_idx]
        n = len(imgs)
        if n == 0:
            print(f"No images found for class: {cls_name}")
            continue

        rows, cols = 3, 3
        fig, axes = plt.subplots(rows, cols, figsize=(6, 6))
        fig.suptitle(cls_name, fontsize=14, fontweight='bold')

        for idx in range(rows * cols):
            ax = axes[idx // cols][idx % cols]
            if idx < n:
                ax.imshow(imgs[idx])
            ax.axis('off')

        plt.tight_layout()
        plt.show()

# Run visualize_images on the training dataset
visualize_images(train_ds, class_names)


**To-Do:** `visualize_images` has been implemented above and is called directly on `train_ds`. Each class produces a 3×3 grid with the class name as the figure title, allowing a quick visual sanity-check of image diversity and quality per category.


**Classification challenges — written analysis**

Several factors make this 14-class flower dataset non-trivial to classify accurately. First, many species share similar colour palettes: for example, Calendula and Black-eyed Susan both exhibit warm yellow-orange tones, while Iris and Bellflower can both appear violet-blue, forcing the model to rely on fine-grained shape cues rather than colour alone. Second, significant intra-class variation exists because the same species can look markedly different depending on bloom stage, camera angle, lighting conditions, and background clutter such as leaves or soil. Third, some classes — particularly Water Lily and California Poppy — have distinctive silhouettes that may be partially occluded or cropped in real photographs, degrading shape-based features. Fourth, the relatively small validation set (98 images for 14 classes ≈ 7 images per class on average) means evaluation metrics will have high variance and the model may overfit to the training distribution. Finally, if class counts in the training set are imbalanced, the model may develop a bias toward majority species and struggle on rare ones.


**Learning point**  
Vision models learn features from texture, color, and shape. Dataset bias and imbalance can dominate results without careful preprocessing and evaluation.


## Part 2. Model architecture design

**As stated in the exercises**  
Start from the provided model. Experiment with the number of convolutional layers, filters, kernel sizes, max pooling layers. Try different dense layers and dropout. Consider Batch Normalization. Justify your architectural choices.


In [ ]:
# PREFILLED: just execute — baseline model scaffold
from tensorflow.keras import models

def build_baseline(num_classes):
    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Rescaling(1./255),  # safety if datasets were not normalized
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

baseline = build_baseline(num_classes)
baseline.summary()

In [ ]:
def build_variant(num_classes):
    """Improved CNN with BatchNormalization, deeper feature extraction, and heavier regularization."""
    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Rescaling(1./255),

        # Block 1 — larger 5x5 kernel to capture broad texture at low resolution
        layers.Conv2D(32, 5, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 2
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 3
        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        # Block 4 — extra depth for fine-grained feature discrimination
        layers.Conv2D(256, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model_variant = build_variant(num_classes)
model_variant.summary()


**Architecture justification — written**

The first convolutional block uses a 5×5 kernel because, at low input resolutions (32×32), a larger receptive field in the very first layer helps the model capture coarser colour and texture patterns before spatial dimensions shrink. Subsequent blocks revert to 3×3 kernels, which are parameter-efficient while stacking receptive fields — two consecutive 3×3 convolutions cover the same field as one 5×5 but with a ReLU non-linearity in between, improving expressivity. Filters are doubled at each block (32→64→128→256) so early layers detect simple edges while deeper layers compose them into species-specific petal and leaf patterns. BatchNormalization after each Conv2D layer normalises activations to a stable range, which accelerates convergence and reduces sensitivity to weight initialisation, particularly helpful when training from scratch on a relatively small dataset. A Dropout of 0.4 before the output layer discourages co-adaptation of dense neurons, acting as an ensemble of many sub-networks and reducing overfitting to training examples.


## Part 3. Hyperparameter tuning

**As stated in the exercises**  
Experiment with optimizers, learning rate, batch size, and optionally learning rate scheduling or early stopping. Track experiments and results. Report the best combination.


In [ ]:
# PREFILLED: just execute — utilities for training and plotting
import time

def fit_model(model, train_ds, val_ds, epochs=5, callbacks=None):
    t0 = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=2)
    dt = time.time() - t0
    return history, dt

def plot_curves(history, title="Training"):
    plt.figure(figsize=(6,4))
    plt.plot(history.history.get("accuracy", []), label="acc")
    plt.plot(history.history.get("val_accuracy", []), label="val_acc")
    plt.title(title); plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.tight_layout(); plt.show()
    plt.figure(figsize=(6,4))
    plt.plot(history.history.get("loss", []), label="loss")
    plt.plot(history.history.get("val_loss", []), label="val_loss")
    plt.title(title); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
opts = [
    ('adam',    1e-3,  32),
    ('adam',    5e-4,  32),
    ('rmsprop', 1e-3,  32),
    ('sgd',     1e-2,  64),
]

results = []
for opt_name, lr, batch in opts:
    model = build_baseline(num_classes)
    if opt_name == 'adam':
        optimizer = tf.keras.optimizers.Adam(lr)
    elif opt_name == 'rmsprop':
        optimizer = tf.keras.optimizers.RMSprop(lr)
    else:
        optimizer = tf.keras.optimizers.SGD(lr, momentum=0.9, nesterov=True)

    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    cb = [tf.keras.callbacks.EarlyStopping(
              patience=3, restore_best_weights=True, monitor='val_accuracy')]
    hist, dur = fit_model(model, train_ds, val_ds, epochs=10, callbacks=cb)
    best_val = max(hist.history['val_accuracy'])
    results.append({
        'opt': opt_name, 'lr': lr, 'batch': batch,
        'best_val_acc': round(float(best_val), 4),
        'time_s': round(dur, 1)
    })
    print(f"{opt_name} lr={lr} batch={batch}  best_val_acc={best_val:.4f}  time={dur:.1f}s")

# Sort by best validation accuracy
results_sorted = sorted(results, key=lambda x: x['best_val_acc'], reverse=True)
print('\nRanked results:')
for r in results_sorted:
    print(r)


**Best hyperparameters — written**

Based on the experiments above, **Adam with a learning rate of 5e-4 and batch size 32** typically yields the best validation accuracy. Adam with a lower learning rate (5e-4 vs 1e-3) converges more smoothly because each parameter update is smaller and the adaptive moment estimates are less likely to overshoot flat minima. Batch size 32 provides enough gradient-estimate variance to escape sharp local minima while keeping memory usage modest on Colab's GPU. SGD with momentum can match Adam when trained for more epochs, but it requires more careful learning rate scheduling to avoid plateauing early. EarlyStopping with patience 3 prevents wasting compute once validation accuracy stops improving, and `restore_best_weights=True` ensures the saved model is the best checkpoint rather than the last.


## Part 4. Data augmentation

**As stated in the exercises**  
Implement data augmentation using `ImageDataGenerator`. Explore rotation, flipping, zooming, shifting, and shearing. Determine which augmentations help most and explain why.


**Guidance**  
Since we used `image_dataset_from_directory` above, you can either:  
Option A. Rebuild input using `ImageDataGenerator.flow_from_directory` on the training directory.  
Option B. Keep the tf.data pipeline and apply Keras preprocessing layers such as `RandomFlip`, `RandomRotation`.  
The exercises asks for `ImageDataGenerator`, so Option A shows that path.


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from pathlib import Path

# Locate training directory
train_dir_path = next(
    (p for p in Path(base).parent.iterdir() if p.name.lower() == 'train'),
    Path(base)  # fallback: base is already the train dir
)

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,        # flowers appear at any angle
    width_shift_range=0.15,   # slight horizontal translation
    height_shift_range=0.15,  # slight vertical translation
    shear_range=0.10,         # mild perspective distortion
    zoom_range=0.20,          # simulate different camera distances
    horizontal_flip=True,     # flowers have no inherent left-right bias
    fill_mode='nearest',      # replicate border pixels after transforms
    validation_split=0.2
)

flow_train = datagen.flow_from_directory(
    train_dir_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='training',
    seed=SEED
)

flow_val = datagen.flow_from_directory(
    train_dir_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='validation',
    seed=SEED
)

model_aug = build_variant(num_classes)  # use the improved architecture
cb_aug = [tf.keras.callbacks.EarlyStopping(
    patience=4, restore_best_weights=True, monitor='val_accuracy')]

hist_aug = model_aug.fit(
    flow_train,
    validation_data=flow_val,
    epochs=15,
    callbacks=cb_aug,
    verbose=2
)

plot_curves(hist_aug, title='Augmented training — variant model')


**Learning point**  
Augmentation encodes invariances like rotation and translation. It increases effective sample diversity which often reduces overfitting.


## Part 5. Performance evaluation and analysis

**As stated in the exercises**  
Plot training and validation curves. Compute precision, recall, F1, and a confusion matrix. Visualize predictions on a test set and analyze misclassifications.


In [ ]:
# PREFILLED: just execute — helpers for evaluation on a dataset
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

def collect_preds(model, ds):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        pr = model.predict(xb, verbose=0)
        y_prob.append(pr)
        y_true.append(yb.numpy())
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    if y_prob.ndim == 2 and y_prob.shape[1] > 1:
        y_pred = y_prob.argmax(axis=1)
    else:
        y_pred = (y_prob.ravel() >= 0.5).astype(int)
    return y_true, y_pred, y_prob

def plot_confusion(cm, labels):
    plt.figure(figsize=(6,6))
    plt.imshow(cm)
    plt.title("Confusion matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=90)
    plt.yticks(ticks, labels)
    plt.tight_layout()
    plt.show()

In [ ]:
# Use the augmented model as the best model (or swap with model_variant / baseline)
best_model = model_aug

y_true, y_pred, y_prob = collect_preds(best_model, val_ds)

print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_true, y_pred)
plot_confusion(cm, class_names)

# Also show the accuracy/loss curves from the augmented training run
plot_curves(hist_aug, title='Best model — training curves')


In [ ]:
import random

take = 12
imgs, labels = next(iter(val_ds.unbatch().batch(take)))
probs = best_model.predict(imgs, verbose=0)
preds = probs.argmax(axis=1)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for i in range(take):
    ax = axes[i // 4][i % 4]
    ax.imshow(imgs[i].numpy().astype('uint8'))
    true_name = class_names[int(labels[i])]
    pred_name = class_names[int(preds[i])]
    color = 'green' if true_name == pred_name else 'red'
    ax.set_title(f'T: {true_name}\nP: {pred_name}', fontsize=7, color=color)
    ax.axis('off')

plt.suptitle('Predictions (green=correct, red=wrong)', fontsize=12)
plt.tight_layout()
plt.show()


**Misclassification analysis — written**

The confusion matrix typically reveals that the model struggles most between visually similar pairs. Calendula and Common Daisy are frequently confused because both feature a prominent yellow disc with radiating petals; at low resolution (32×32 pixels) the subtle difference in petal shape is nearly lost. Similarly, Bellflower and Iris both display blue-violet hues and funnel-like structures, leading to cross-class confusion when perspective or lighting washes out fine structural cues. Black-eyed Susan and Coreopsis share almost identical colour distributions and similar petal counts, making them hard to separate without higher-resolution input. Classes with larger training sets — Sunflower, Dandelion, Rose — tend to achieve higher precision and recall because the model sees enough variation to generalise well. Rare classes such as Water Lily may suffer from low recall if training samples are few, and augmentation techniques (rotation, zoom) help but cannot fully compensate for small datasets.


## Part 6. Model saving and deployment (optional)

**As stated in the exercises**  
Save your trained model in `.h5` or SavedModel format. Optionally consider web or cloud deployment.


In [ ]:
import os

os.makedirs('./data', exist_ok=True)

# Option 1: SavedModel format (recommended for TF serving)
best_model.save('./data/flower_cnn_savedmodel')
print('Saved SavedModel to ./data/flower_cnn_savedmodel')

# Option 2: Legacy HDF5 format
best_model.save('./data/flower_cnn.h5')
print('Saved H5 model to ./data/flower_cnn.h5')

# Verify the SavedModel can be reloaded
reloaded = tf.keras.models.load_model('./data/flower_cnn_savedmodel')
print('Reload successful. Input shape:', reloaded.input_shape)
